<a href="https://colab.research.google.com/github/kotok9/app-stat-lab01/blob/main/lab01_asteroids.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab01: доп. задание — сближения астероидов с Землей

In [1]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [2]:
#@title Стиль графиков — запустить один раз в начале { display-mode: "form" }
import matplotlib as mpl
import matplotlib.pyplot as plt

# Палитра «космос»: имена переменных не менять, на них опираются остальные ноутбуки
C_DATA   = "#1F4E8C"   # данные — глубокий синий ночного неба
C_THEORY = "#E85D04"   # теоретическая кривая — раскалённый оранжевый болида
C_ACCENT = "#9B5DE5"   # выделение — фиолетовый туманности
C_MUTED  = "#8A94A6"   # второстепенные элементы — серо-голубая дымка
C_PALETTE = [C_DATA, C_THEORY, C_ACCENT, C_MUTED, "#2A9D8F", "#C9184A"]

# Блок Б: размер и разрешение всех картинок
mpl.rcParams.update({
    "figure.figsize": (9, 5.2),    # размер картинки в дюймах
    "figure.dpi": 110,             # качество на экране
    "savefig.dpi": 150,            # качество при сохранении в файл
    "figure.facecolor": "white",   # белый фон вокруг осей
    "axes.facecolor": "white",     # белый фон внутри осей
    "savefig.facecolor": "white",
})

# Блок В: шрифты
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
})

# Блок Г: рамки и сетка
mpl.rcParams.update({
    "axes.edgecolor": "#B9C0CC",
    "axes.linewidth": 1.0,
    "axes.spines.top": False,     # убрать верхнюю рамку
    "axes.spines.right": False,   # убрать правую рамку
    "axes.grid": True,
    "axes.axisbelow": True,       # сетка под данными, а не поверх
    "grid.color": "#D9DEE7",
    "grid.linewidth": 0.8,
    "grid.alpha": 0.7,
})

# Блок Д: деления на осях
mpl.rcParams.update({
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "xtick.color": "#4A5568",
    "ytick.color": "#4A5568",
})

# Блок Е: линии, легенда и порядок цветов
mpl.rcParams.update({
    "lines.linewidth": 2.0,
    "lines.markersize": 6,
    "legend.frameon": False,      # легенда без рамки
    "legend.fontsize": 10,
    "axes.prop_cycle": mpl.cycler(color=C_PALETTE),  # цвета серий по умолчанию
})

print("Стиль применён. Палитра:", C_PALETTE)

Стиль применён. Палитра: ['#1F4E8C', '#E85D04', '#9B5DE5', '#8A94A6', '#2A9D8F', '#C9184A']


## Extra dataset (сближения астероидов)

### Загрузка данных

In [3]:
URL = "https://ssd-api.jpl.nasa.gov/cad.api?date-min=2015-01-01&date-max=2025-12-31&dist-max=0.05"
j = requests.get(URL).json()
df = pd.DataFrame(j["data"], columns=j["fields"])
for c in ["jd", "dist", "dist_min", "dist_max", "v_rel", "v_inf", "t_sigma_f", "h"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["cd"] = pd.to_datetime(df["cd"], format="%Y-%b-%d %H:%M")
print(df.shape)
df.head()

(14590, 11)


,des,orbit_id,jd,cd,dist,dist_min,dist_max,v_rel,v_inf,t_sigma_f,h
0,2022 AP1,3,2.457024e+06,2015-01-01 00:27:00,0.045100,0.013778,0.076571,11.891749,11.886780,NaN,28.39
1,2015 AE45,10,2.457025e+06,2015-01-02 15:56:00,0.048239,0.048220,0.048258,7.065686,7.057864,NaN,25.30
2,613286,43,2.457025e+06,2015-01-02 21:46:00,0.026522,0.026522,0.026522,12.703378,12.695467,NaN,20.63
3,2014 YE42,11,2.457026e+06,2015-01-03 15:00:00,0.010995,0.010971,0.011020,13.998882,13.981561,NaN,23.40
4,2014 YB42,6,2.457027e+06,2015-01-04 05:33:00,0.042573,0.042222,0.042925,9.659652,9.653170,NaN,25.40


Данные загружены с сайта NASA. Каждая строка таблицы это одно сближение астероида с Землей. В таблице 14 590 строк и 11 столбцов.

In [4]:
print("всего записей:", len(df))
print(df[["dist", "v_rel", "h"]].notna().sum())
print("пар (v_rel, dist) без пропусков:", df[["v_rel", "dist"]].dropna().shape[0])
print("пар (v_rel, h) без пропусков:", df[["v_rel", "h"]].dropna().shape[0])

всего записей: 14590
dist     14590
v_rel    14590
h        14588
dtype: int64
пар (v_rel, dist) без пропусков: 14590
пар (v_rel, h) без пропусков: 14588


Выше показано сколько непустых значений в столбцах dist, v_rel и h. Считаем только по тем значениям которые есть.